<a href="https://colab.research.google.com/github/arun-antony/ai-colab-tryout/blob/main/tcs_ai_system_case_study_stock_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import numpy as np

### 1. Data Loading

In [2]:
# Define the path to your Excel file
excel_file_path = '/content/data/robinhood_style_stock_activity_case_study.xlsx'

# Read the Excel file into a pandas DataFrame
df = pd.read_excel(excel_file_path)

# Display the first 5 rows of the DataFrame
display(df.head())

,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount
0,2021-01-05,2021-01-05,2021-01-05,NaN,ACH Deposit,ACH,NaN,NaN,2500.00
1,2021-01-18,2021-01-18,2021-01-20,META,Market Buy 4.6606 shares of Meta Platforms Inc.,Buy,4.6606,206.74,-963.53
2,2021-02-03,2021-02-03,2021-02-05,MSFT,Market Buy 1.5733 shares of Microsoft Corporation,Buy,1.5733,200.01,-314.68
3,2021-02-28,2021-02-28,2021-03-02,AMD,Market Buy 10.6469 shares of Advanced Micro De...,Buy,10.6469,98.73,-1051.17
4,2021-03-07,2021-03-07,2021-03-09,MSFT,Market Buy 4.308 shares of Microsoft Corporation,Buy,4.3080,254.32,-1095.61


### 2. Data Cleaning
1. Remove all rows that are not a trading transaction. Eg: Deposit, Interest payout etc. Create a filter list that decides what kind of transaction to be filtered.
2. remove transactions that does not have a date or date is invalid
3. remove transactions that does not have a quantity
4. remove transactions that does not have a price




In [3]:
#@title Flags that control what kind of data rows need to be removed or cleaned { run: "auto" }
apply_transaction_code_filter = True #@param {type:"boolean"}
apply_invalid_date_filter = True #@param {type:"boolean"}
apply_invalid_quantity_filter = True #@param {type:"boolean"}
apply_invalid_price_filter = True #@param {type:"boolean"}

In [4]:
# Create a copy to avoid SettingWithCopyWarning
cleaned_df = df.copy()

if apply_transaction_code_filter:
    # 1.1 Removes rows where 'Trans Code' is missing completely
    cleaned_df = cleaned_df.dropna(subset=['Trans Code'])

    # List of transaction codes that will be allowedd for processing
    # Note: Use lower cased values in this filter list
    transaction_code_allow_filter = ['buy', 'sell']

    # 1.2 Apply filter so that only allowed transactions are retained
    cleaned_df = cleaned_df[cleaned_df['Trans Code'].str.lower().isin(transaction_code_allow_filter)]

    # lower case all the tranaction codes
    cleaned_df['Trans Code'] = cleaned_df['Trans Code'].str.lower()

if apply_invalid_date_filter:
    date_columns = ['Activity Date', 'Process Date', 'Settle Date']
    # 2.1 Drop rows where any of the date is NaN
    cleaned_df = cleaned_df.dropna(subset=date_columns, how='any')

    # 2.2 Convert all 3 columns to datetime, turning unparseable dates into NaT (Not a Time)
    for col in date_columns:
        cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')

    # 2.3 Drop rows again if the conversion generated any new NaT values
    cleaned_df = cleaned_df.dropna(subset=date_columns, how='any')

if apply_invalid_quantity_filter:
    # 3.1 Drop rows where 'Quantity' is NaN
    cleaned_df = cleaned_df.dropna(subset=['Quantity'])

if apply_invalid_price_filter:
    # 4.1 Drop rows where 'Price' is NaN
    cleaned_df = cleaned_df.dropna(subset=['Price'])


cleaned_df.head()

,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount
1,2021-01-18,2021-01-18,2021-01-20,META,Market Buy 4.6606 shares of Meta Platforms Inc.,buy,4.6606,206.74,-963.53
2,2021-02-03,2021-02-03,2021-02-05,MSFT,Market Buy 1.5733 shares of Microsoft Corporation,buy,1.5733,200.01,-314.68
3,2021-02-28,2021-02-28,2021-03-02,AMD,Market Buy 10.6469 shares of Advanced Micro De...,buy,10.6469,98.73,-1051.17
4,2021-03-07,2021-03-07,2021-03-09,MSFT,Market Buy 4.308 shares of Microsoft Corporation,buy,4.3080,254.32,-1095.61
5,2021-03-10,2021-03-10,2021-03-12,PLTR,Market Buy 5.54 shares of Palantir Technologie...,buy,5.5400,15.03,-83.27


###3. Core Metric Engineering
The following metrics are calculated in this funtion:
1. Total Cost Basis
2. Per Unit Cost Basis
3. Realized PnL
4. Holding Period
5. Win or Loss

In [53]:
def _process_buy_transaction(qty, price, activity_date, ticker, inventory):
    """Adds shares to inventory and returns blank performance metrics."""
    inventory[ticker].append({
        'qty': qty,
        'price': price,
        'act_date': activity_date
    })

    return {
        'per_unit_cost': price,
        'total_cost': price * qty,
        'realized_pnl': np.nan,
        'return_percentage': np.nan,
        'holding_days': np.nan,
        'holding_details': [],
        'win_loss_status': 'NA'
    }

In [46]:
def _process_sell_transaction(qty, price, activity_date, ticker, inventory):
    """Executes FIFO queue matching and calculates realized performance."""
    shares_to_sell = qty
    total_cost_of_shares_sold = 0.0
    total_weighted_holding_days = 0.0
    lot_breakdown = []

    while shares_to_sell > 0 and len(inventory.get(ticker, [])) > 0:
        current_lot = inventory[ticker][0]
        lot_qty, lot_price = current_lot['qty'], current_lot['price']
        days_held = (activity_date - current_lot['act_date']).days

        if lot_qty <= shares_to_sell:
            total_cost_of_shares_sold += lot_qty * lot_price
            total_weighted_holding_days += lot_qty * days_held
            lot_breakdown.append({'qty': lot_qty, 'days': days_held})

            shares_to_sell -= lot_qty
            inventory[ticker].pop(0)
        else:
            total_cost_of_shares_sold += shares_to_sell * lot_price
            total_weighted_holding_days += shares_to_sell * days_held
            current_lot['qty'] = round(current_lot['qty'] - shares_to_sell, 4)
            lot_breakdown.append({'qty': shares_to_sell, 'days': days_held})
            shares_to_sell = 0

    # Fallback for unmatched shares
    if shares_to_sell > 0:
        total_cost_of_shares_sold += shares_to_sell * price
        lot_breakdown.append({'qty': shares_to_sell, 'days': 0, 'note': 'unmatched'})

    # Compute final metrics
    realized_pnl = (qty * price) - total_cost_of_shares_sold
    return_pct = (100 * realized_pnl / total_cost_of_shares_sold) if total_cost_of_shares_sold > 0 else 0.0

    return {
        'per_unit_cost': total_cost_of_shares_sold / qty,
        'total_cost': total_cost_of_shares_sold,
        'realized_pnl': realized_pnl,
        'return_percentage': return_pct,
        'holding_days': total_weighted_holding_days / qty,
        'holding_details': lot_breakdown,
        'win_loss_status': 'Win' if realized_pnl > 0 else 'Loss' if realized_pnl < 0 else 'Flat'
    }

In [50]:
def derive_core_metrics(df):
    processed_df = df.sort_values(by=['Activity Date', 'Process Date']).copy()
    inventory = {}

    # We will collect the dictionaries returned by the helpers here
    results_map = {}

    for idx, row in processed_df.iterrows():
        ticker = row['Instrument']
        trans_type = row['Trans Code'].strip().lower()
        qty = abs(float(row['Quantity']))
        price = float(row['Price'])
        activity_date = row['Activity Date']

        if ticker not in inventory:
            inventory[ticker] = []

        if trans_type == 'buy':
            results_map[idx] = _process_buy_transaction(qty, price, activity_date, ticker, inventory)
        elif trans_type == 'sell':
            results_map[idx] = _process_sell_transaction(qty, price, activity_date, ticker, inventory)
        else:
            # Create a blank fallback dictionary
            results_map[idx] = {'per_unit_cost': 0.0, 'total_cost': 0.0, 'realized_pnl': 0.0,
                                'return_percentage': 0.0, 'holding_days': np.nan, 'holding_details': [], 'win_loss_status': 'NA'}

    # Map the collected dictionary keys to new columns efficiently
    for key in ['per_unit_cost', 'total_cost', 'realized_pnl', 'return_percentage', 'holding_days', 'holding_details', 'win_loss_status']:
        processed_df[key] = processed_df.index.map(lambda i: results_map[i][key])

    return processed_df

In [51]:
def derive_core_metrics_old(df):
    """
    Calculates the running cost basis and realized PnL for each transaction
    using the FIFO (First-In, First-Out) accounting method.
    """
    # 1. Ensure data is sorted chronologically by dates
    processed_df = df.sort_values(by=['Activity Date', 'Process Date']).copy()

    # 2. Initialize the new tracking columns with 0.0
    # Use native Python dictionaries to collect results (prevents performance lag & warnings)
    per_unit_cost_basis_results = {}
    total_cost_basis_results = {}
    realized_pnl_results = {}
    return_percentage_results = {}
    holding_period_results = {}
    win_or_loss_results = {}
    holding_details_results = {} # hold a list of split holding days details

    # 3. Ledger to track open positions per instrument:
    # Structure: { 'TICKER': [[quantity_lot_1, price_lot_1, date_lote_1], [quantity_lot_2, price_lot_2, date_lote_2]] }
    inventory = {}

    # 4. Iterate through rows safely using .iterrows() or .itertuples()
    for idx, row in processed_df.iterrows():
        ticker = row['Instrument']
        trans_type = row['Trans Code'].strip().lower()
        qty = abs(float(row['Quantity'])) # This is negative for sell, hence taking the abs
        price = float(row['Price'])
        activity_date = row['Activity Date']

        # Initialize tracking queue for a ticker if it doesn't exist
        if ticker not in inventory:
            inventory[ticker] = []

        if trans_type == 'buy':
            # Store as a mutable dictionary lot
            inventory[ticker].append({'qty': qty, 'price': price, 'act_date': activity_date})

            # For a buy, the effective cost basis of the transaction is what you paid for it
            per_unit_cost_basis_results[idx] = price
            total_cost_basis_results[idx] = (price * qty)
            realized_pnl_results[idx] = 0.0
            return_percentage_results[idx] = 0.0
            holding_period_results[idx] = np.nan
            holding_details_results[idx] = [] #NA for buys
            win_or_loss_results[idx] = 'NA'

        elif trans_type == 'sell':
            shares_to_sell = qty
            total_cost_of_shares_sold = 0.0
            total_weighted_holding_days = 0.0
            lot_breakdown = []

            # Dequeue from inventory using FIFO logic
            while shares_to_sell > 0 and len(inventory[ticker]) > 0:
                current_lot = inventory[ticker][0] # Look at the oldest available lot
                lot_qty, lot_price = current_lot['qty'], current_lot['price']
                lot_date = current_lot['act_date']
                days_held = (activity_date - lot_date).days

                if lot_qty <= shares_to_sell:
                    # Eat up the entire current lot
                    total_cost_of_shares_sold += lot_qty * lot_price

                    total_weighted_holding_days += lot_qty * days_held
                    # Log this specific piece of the trade
                    lot_breakdown.append({'qty': lot_qty, 'days': days_held})
                    shares_to_sell -= lot_qty
                    inventory[ticker].pop(0) # Remove completely filled lot
                else:
                    # Partially eat up the current lot
                    total_cost_of_shares_sold += shares_to_sell * lot_price
                    # for the below calculation apply rounding to 4 digits
                    current_lot['qty'] = round(current_lot['qty'] - shares_to_sell, 4) # Deduct remaining shares from lot

                    total_weighted_holding_days += shares_to_sell * days_held

                    # Log this specific piece of the trade
                    lot_breakdown.append({'qty': shares_to_sell, 'days': days_held})

                    shares_to_sell = 0

            # Handle edge case: Sold short or missing history data
            if shares_to_sell > 0:
                # If no matching buy lots remain, fall back to current transaction price
                total_cost_of_shares_sold += shares_to_sell * price
                lot_breakdown.append({'qty': shares_to_sell, 'days': 0, 'note': 'short/unmatched'})

            # Calculate metrics for the transaction row
            effective_total_cost_basis = total_cost_of_shares_sold / qty
            total_revenue = qty * price
            realized_pnl = total_revenue - total_cost_of_shares_sold
            weighted_holding_days = total_weighted_holding_days / qty

            # Populate the output columns
            per_unit_cost_basis_results[idx] = effective_total_cost_basis
            total_cost_basis_results[idx] = total_cost_of_shares_sold
            realized_pnl_results[idx] = realized_pnl
            return_percentage_results[idx] = (100 * realized_pnl / total_cost_of_shares_sold) if total_cost_of_shares_sold > 0 else 0.0
            holding_period_results[idx] = weighted_holding_days
            holding_details_results[idx] = lot_breakdown
            win_or_loss_results[idx] = 'Win' if realized_pnl > 0 else 'Loss' if realized_pnl < 0 else 'NA'

        else:
            # Fallback for any unexpected non-trade types remaining
            per_unit_cost_basis_results[idx] = 0.0
            total_cost_basis_results[idx] = 0.0
            realized_pnl_results[idx] = 0.0
            return_percentage_results[idx] = 0.0
            holding_period_results[idx] = np.nan
            holding_details_results[idx] = [] #NA for buys
            win_or_loss_results[idx] = 'NA'

    # 4. Map dictionaries back to the dataframe all at once (Highly efficient)
    processed_df['Per Unit Cost Basis'] = processed_df.index.map(per_unit_cost_basis_results)
    processed_df['Total Cost Basis'] = processed_df.index.map(total_cost_basis_results)
    processed_df['Realized_PnL'] = processed_df.index.map(realized_pnl_results)
    processed_df['Return Percentage'] = processed_df.index.map(return_percentage_results)
    processed_df['Weighted Holding Period'] = processed_df.index.map(holding_period_results)
    processed_df['Holding Details'] = processed_df.index.map(holding_details_results)
    processed_df['Win or Loss'] = processed_df.index.map(win_or_loss_results)

    return processed_df



In [54]:
test = derive_core_metrics(cleaned_df.copy())
test.head(10)

,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount,per_unit_cost,total_cost,realized_pnl,return_percentage,holding_days,holding_details,win_loss_status
1,2021-01-18,2021-01-18,2021-01-20,META,Market Buy 4.6606 shares of Meta Platforms Inc.,buy,4.6606,206.74,-963.53,206.740000,963.532444,NaN,NaN,NaN,[],NA
2,2021-02-03,2021-02-03,2021-02-05,MSFT,Market Buy 1.5733 shares of Microsoft Corporation,buy,1.5733,200.01,-314.68,200.010000,314.675733,NaN,NaN,NaN,[],NA
3,2021-02-28,2021-02-28,2021-03-02,AMD,Market Buy 10.6469 shares of Advanced Micro De...,buy,10.6469,98.73,-1051.17,98.730000,1051.168437,NaN,NaN,NaN,[],NA
4,2021-03-07,2021-03-07,2021-03-09,MSFT,Market Buy 4.308 shares of Microsoft Corporation,buy,4.3080,254.32,-1095.61,254.320000,1095.610560,NaN,NaN,NaN,[],NA
5,2021-03-10,2021-03-10,2021-03-12,PLTR,Market Buy 5.54 shares of Palantir Technologie...,buy,5.5400,15.03,-83.27,15.030000,83.266200,NaN,NaN,NaN,[],NA
6,2021-03-15,2021-03-15,2021-03-17,AMD,Market Buy 8.8108 shares of Advanced Micro Dev...,buy,8.8108,81.08,-714.38,81.080000,714.379664,NaN,NaN,NaN,[],NA
7,2021-03-20,2021-03-20,2021-03-22,AMD,Market Sell 8.8108 shares of Advanced Micro De...,sell,-8.8108,84.47,744.25,98.730000,869.890284,-125.642008,-14.443432,20.000000,"[{'qty': 8.8108, 'days': 20}]",Loss
8,2021-03-21,2021-03-21,2021-03-23,MSFT,Market Sell 4.308 shares of Microsoft Corporation,sell,-4.3080,237.07,1021.30,234.485756,1010.164637,11.132923,1.102090,25.686537,"[{'qty': 1.5733, 'days': 46}, {'qty': 2.7347, ...",Win
9,2021-03-24,2021-03-24,2021-03-26,GOOGL,Market Buy 11.1618 shares of Alphabet Inc.,buy,11.1618,88.07,-983.02,88.070000,983.019726,NaN,NaN,NaN,[],NA
11,2021-04-04,2021-04-04,2021-04-06,MSFT,Market Sell 1.5733 shares of Microsoft Corpora...,sell,-1.5733,208.67,328.30,254.320000,400.121656,-71.821145,-17.949827,28.000000,"[{'qty': 1.5733, 'days': 28}]",Loss


In [41]:
# Testing derive profit and loss
test = derive_core_metrics(cleaned_df.copy())
test.head(10)

,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount,Per Unit Cost Basis,Total Cost Basis,Realized_PnL,Return Percentage,Weighted Holding Period,Holding Details,Win or Loss
1,2021-01-18,2021-01-18,2021-01-20,META,Market Buy 4.6606 shares of Meta Platforms Inc.,buy,4.6606,206.74,-963.53,206.740000,963.532444,0.000000,0.000000,NaN,[],NA
2,2021-02-03,2021-02-03,2021-02-05,MSFT,Market Buy 1.5733 shares of Microsoft Corporation,buy,1.5733,200.01,-314.68,200.010000,314.675733,0.000000,0.000000,NaN,[],NA
3,2021-02-28,2021-02-28,2021-03-02,AMD,Market Buy 10.6469 shares of Advanced Micro De...,buy,10.6469,98.73,-1051.17,98.730000,1051.168437,0.000000,0.000000,NaN,[],NA
4,2021-03-07,2021-03-07,2021-03-09,MSFT,Market Buy 4.308 shares of Microsoft Corporation,buy,4.3080,254.32,-1095.61,254.320000,1095.610560,0.000000,0.000000,NaN,[],NA
5,2021-03-10,2021-03-10,2021-03-12,PLTR,Market Buy 5.54 shares of Palantir Technologie...,buy,5.5400,15.03,-83.27,15.030000,83.266200,0.000000,0.000000,NaN,[],NA
6,2021-03-15,2021-03-15,2021-03-17,AMD,Market Buy 8.8108 shares of Advanced Micro Dev...,buy,8.8108,81.08,-714.38,81.080000,714.379664,0.000000,0.000000,NaN,[],NA
7,2021-03-20,2021-03-20,2021-03-22,AMD,Market Sell 8.8108 shares of Advanced Micro De...,sell,-8.8108,84.47,744.25,98.730000,869.890284,-125.642008,-14.443432,20.000000,"[{'qty': 8.8108, 'days': 20}]",Loss
8,2021-03-21,2021-03-21,2021-03-23,MSFT,Market Sell 4.308 shares of Microsoft Corporation,sell,-4.3080,237.07,1021.30,234.485756,1010.164637,11.132923,1.102090,25.686537,"[{'qty': 1.5733, 'days': 46}, {'qty': 2.7347, ...",Win
9,2021-03-24,2021-03-24,2021-03-26,GOOGL,Market Buy 11.1618 shares of Alphabet Inc.,buy,11.1618,88.07,-983.02,88.070000,983.019726,0.000000,0.000000,NaN,[],NA
11,2021-04-04,2021-04-04,2021-04-06,MSFT,Market Sell 1.5733 shares of Microsoft Corpora...,sell,-1.5733,208.67,328.30,254.320000,400.121656,-71.821145,-17.949827,28.000000,"[{'qty': 1.5733, 'days': 28}]",Loss
